# 3.2 Code Brief: Building Tree-Based Models in Practice

This notebook contains a condensed reference of the key code patterns from notebook 3.2. Use it as a quick reference.

## Key Pattern: Instantiate → Fit → Predict

```python
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Same three lines for any model:
model = ModelClass(**params)
model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:, 1]
```

## Setup and Data Preparation

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report)
from sklearn.model_selection import cross_val_score, StratifiedKFold
import time

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# Set up file paths
project_path = '/content/drive/MyDrive/Applied-Data-Analytics-For-Higher-Education-Course-3'
data_filepath = '/data/'

train_df = pd.read_csv(f'{project_path}{data_filepath}training.csv')
test_df = pd.read_csv(f'{project_path}{data_filepath}testing.csv')
train_df['DEPARTED'] = (train_df['SEM_3_STATUS'] != 'E').astype(int)
test_df['DEPARTED'] = (test_df['SEM_3_STATUS'] != 'E').astype(int)

print(f"Training set: {train_df.shape[0]:,} students")
print(f"Testing set: {test_df.shape[0]:,} students")

In [ ]:
# NO SCALING NEEDED for tree-based models!
numeric_features = [
    'HS_GPA', 'HS_MATH_GPA', 'HS_ENGL_GPA',
    'UNITS_ATTEMPTED_1', 'UNITS_ATTEMPTED_2',
    'UNITS_COMPLETED_1', 'UNITS_COMPLETED_2',
    'DFW_UNITS_1', 'DFW_UNITS_2',
    'GPA_1', 'GPA_2',
    'DFW_RATE_1', 'DFW_RATE_2',
    'GRADE_POINTS_1', 'GRADE_POINTS_2']
categorical_features = ['RACE_ETHNICITY', 'GENDER', 'FIRST_GEN_STATUS', 'COLLEGE']
target = 'DEPARTED'

train_encoded = pd.get_dummies(train_df[numeric_features + categorical_features],
                               columns=categorical_features, drop_first=True)
test_encoded = pd.get_dummies(test_df[numeric_features + categorical_features],
                              columns=categorical_features, drop_first=True)
train_encoded, test_encoded = train_encoded.align(test_encoded, join='left', axis=1, fill_value=0)

X_train = train_encoded
y_train = train_df[target]
X_test = test_encoded
y_test = test_df[target]

print(f"Features: {X_train.shape[1]}")

## Model 1: Decision Tree

In [ ]:
dt = DecisionTreeClassifier(
    max_depth=8,
    min_samples_split=20,
    min_samples_leaf=10,
    max_features='sqrt',
    class_weight='balanced',
    random_state=RANDOM_STATE
)
start = time.time()
dt.fit(X_train, y_train)
dt_time = time.time() - start
dt_pred = dt.predict(X_test)
dt_prob = dt.predict_proba(X_test)[:, 1]

print("=== Decision Tree Results ===")
print(f"Accuracy:  {accuracy_score(y_test, dt_pred):.4f}")
print(f"Precision: {precision_score(y_test, dt_pred):.4f}")
print(f"Recall:    {recall_score(y_test, dt_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, dt_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, dt_prob):.4f}")

## Model 2: Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    class_weight='balanced',
    n_jobs=-1,
    random_state=RANDOM_STATE
)
start = time.time()
rf.fit(X_train, y_train)
rf_time = time.time() - start
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print("=== Random Forest Results ===")
print(f"Accuracy:  {accuracy_score(y_test, rf_pred):.4f}")
print(f"Precision: {precision_score(y_test, rf_pred):.4f}")
print(f"Recall:    {recall_score(y_test, rf_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, rf_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, rf_prob):.4f}")

## Model 3: XGBoost

In [ ]:
xgb = XGBClassifier(
    n_estimators=150,
    learning_rate=0.1,
    max_depth=5,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=len(y_train[y_train==0]) / len(y_train[y_train==1]),
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=RANDOM_STATE
)
start = time.time()
xgb.fit(X_train, y_train)
xgb_time = time.time() - start
xgb_pred = xgb.predict(X_test)
xgb_prob = xgb.predict_proba(X_test)[:, 1]

print("=== XGBoost Results ===")
print(f"Accuracy:  {accuracy_score(y_test, xgb_pred):.4f}")
print(f"Precision: {precision_score(y_test, xgb_pred):.4f}")
print(f"Recall:    {recall_score(y_test, xgb_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, xgb_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, xgb_prob):.4f}")

## Side-by-Side Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest', 'XGBoost'],
    'Accuracy': [accuracy_score(y_test, p) for p in [dt_pred, rf_pred, xgb_pred]],
    'Precision': [precision_score(y_test, p) for p in [dt_pred, rf_pred, xgb_pred]],
    'Recall': [recall_score(y_test, p) for p in [dt_pred, rf_pred, xgb_pred]],
    'F1 Score': [f1_score(y_test, p) for p in [dt_pred, rf_pred, xgb_pred]],
    'ROC-AUC': [roc_auc_score(y_test, p) for p in [dt_prob, rf_prob, xgb_prob]],
    'Train Time (s)': [dt_time, rf_time, xgb_time]
})
print("=" * 80)
print("TREE-BASED MODELS: HEAD-TO-HEAD COMPARISON")
print("=" * 80)
print(results.to_string(index=False))
print("=" * 80)

## Feature Importance

In [ ]:
colors = ['#2ecc71', '#3498db', '#e74c3c']  # Green for DT, Blue for RF, Red for XGB
importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Decision Tree': dt.feature_importances_,
    'Random Forest': rf.feature_importances_,
    'XGBoost': xgb.feature_importances_
})
importance_df = importance_df.sort_values('Random Forest', ascending=False).head(15)

fig = make_subplots(rows=1, cols=3, subplot_titles=('Decision Tree', 'Random Forest', 'XGBoost'))
for col_idx, model_name in enumerate(['Decision Tree', 'Random Forest', 'XGBoost'], 1):
    sorted_df = importance_df.sort_values(by=model_name, ascending=True)
    fig.add_trace(go.Bar(
        y=sorted_df['Feature'], x=sorted_df[model_name],
        orientation='h', marker_color=colors[col_idx-1], showlegend=False
    ), row=1, col=col_idx)
fig.update_layout(height=500, title_text='Top 15 Features by Importance (All Three Models)')
fig.show()

## Visualizing the Decision Tree

In [ ]:
import matplotlib.pyplot as plt

dt_visual = DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=RANDOM_STATE)
dt_visual.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(dt_visual, feature_names=X_train.columns, class_names=['Enrolled', 'Departed'],
          filled=True, rounded=True, fontsize=10, ax=ax)
plt.title('Decision Tree for Student Departure Prediction (max_depth=3)', fontsize=16)
plt.tight_layout()
plt.show()

## Cross-Validation: More Robust Comparison

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scale_pos_weight_value = len(y_train[y_train==0]) / len(y_train[y_train==1])

models_cv = {
    'Decision Tree': DecisionTreeClassifier(
        max_depth=8, min_samples_split=20, min_samples_leaf=10,
        max_features='sqrt', class_weight='balanced', random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_split=10, min_samples_leaf=5,
        max_features='sqrt', class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE
    ),
    'XGBoost': XGBClassifier(
        n_estimators=150, learning_rate=0.1, max_depth=5, min_child_weight=3,
        subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale_pos_weight_value,
        use_label_encoder=False, eval_metric='logloss', random_state=RANDOM_STATE
    )
}

print("5-Fold Cross-Validation Results (F1 Score):")
print("-" * 60)
for name, model in models_cv.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1', n_jobs=-1)
    print(f"{name:20s}: {scores.mean():.4f} (+/- {scores.std():.4f})")
print("-" * 60)

## Key Takeaways

- Same workflow for all three: `instantiate → fit → predict`
- No preprocessing/scaling needed for tree-based models
- All three provide built-in feature importance scores
- Decision Trees are visualizable — great for stakeholder communication
- Random Forests are a robust default; XGBoost often wins on raw performance